# Load Traced Artifacts and Predict


This notebook reads the persisted configs from the Torch example, reloads the saved splits and trained model checkpoint from `data/torch_classification/`, and makes a tiny prediction.

In [1]:
from collections import defaultdict
from pathlib import Path
from pprint import pprint

import torch

from research_pipelines.backends.pickle_backend import PickleBackend

/opt/homebrew/anaconda3/envs/research_pipelines/lib/python3.10/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [2]:
project_root = Path.cwd()
if not (project_root / 'data').exists() and (project_root.parent / 'data').exists():
    project_root = project_root.parent

artifact_root = project_root / 'data' / 'torch_classification'
backend = PickleBackend(directory=str(artifact_root / 'traced_configs'))
configs = backend.load_all()

print(f'Loaded {len(configs)} traced objects from the pickle backend.')
counts = defaultdict(int)
for object_id, entry in configs.items():
    counts[entry['type']] += 1
    print(object_id, entry['type'], entry['config'])

print('')
print('Counts by type:', dict(counts))

Loaded 6 traced objects from the pickle backend.
40dbc913-6673-413d-9595-17edcb98d082 model {'learning_rate': 0.05, 'epochs': 20}
704f83df-9052-4ed2-9824-05fde3917f7f model {'hidden_dim': 8}
c243897a-cb5f-40ce-a3ac-87cc6088e57e dataset {}
eb7af897-9bf2-481f-b609-c0d1441d4ca3 evaluation {}
59633c95-ed5a-46eb-b7d9-639e1566806d dataset {}
03a7e854-249c-4ae5-a7e6-c390d74bc2d5 dataset {}

Counts by type: {'model': 2, 'dataset': 3, 'evaluation': 1}


In [3]:
class SimpleClassifier(torch.nn.Module):
    def __init__(self, input_dim: int = 2, hidden_dim: int = 8):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, 2),
        )

    def forward(self, inputs):
        return self.net(inputs)


def load_tensor_split(path: Path):
    return torch.load(path, map_location='cpu')


def load_classifier_from_checkpoint(checkpoint_path: str):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    model = SimpleClassifier(
        input_dim=int(checkpoint['input_dim']),
        hidden_dim=int(checkpoint['hidden_dim']),
    )
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    return model


def accuracy_for_model(model, features, labels):
    with torch.no_grad():
        logits = model(features)
        predictions = logits.argmax(dim=1)
        return (predictions == labels).float().mean().item()

In [ ]:
split_entries = [entry for entry in configs.values() if entry['type'] == 'dataset']
model_entries = [entry for entry in configs.values() if entry['type'] == 'model' and 'epochs' in entry['config']]

train_split = load_tensor_split(artifact_root / 'splits' / 'train.pt')
val_split = load_tensor_split(artifact_root / 'splits' / 'val.pt')
test_split = load_tensor_split(artifact_root / 'splits' / 'test.pt')

trained_model_entry = model_entries[0]
checkpoint_path = artifact_root / 'checkpoints' / 'simple_classifier.pt'
model = load_classifier_from_checkpoint(str(checkpoint_path))

print('Train split:')
pprint({k: v if k not in {'features', 'labels'} else tuple(v.shape) for k, v in train_split.items()})
print('')
print('Validation split:')
pprint({k: v if k not in {'features', 'labels'} else tuple(v.shape) for k, v in val_split.items()})
print('')
print('Test split:')
pprint({k: v if k not in {'features', 'labels'} else tuple(v.shape) for k, v in test_split.items()})
print('')
print('Traced split artifacts:', len(split_entries))
print('Traced trained model config:')
pprint(trained_model_entry['config'])
print('Model checkpoint:')
print(checkpoint_path)

IndexError: list index out of range

In [ ]:
test_accuracy = accuracy_for_model(model, test_split['features'], test_split['labels'])

sample = test_split['features'][0]
with torch.no_grad():
    logits = model(sample.unsqueeze(0))
    probabilities = torch.softmax(logits, dim=1)[0]
    predicted_class = int(probabilities.argmax().item())

prediction = {
    'predicted_class': predicted_class,
    'confidence': round(float(probabilities[predicted_class].item()), 4),
    'test_accuracy': round(test_accuracy, 4),
}

print('Test accuracy:', round(test_accuracy, 4))
print('Prediction:', prediction)
prediction